# Huấn luyện YOLO11 trên Colab — bài tập Object Detection và Segmentation

Học viên: Võ Hoàng Khang — MSHV 25C01034

Notebook này **không chứa code huấn luyện**; nó clone phần code cấu hình từ repo và gọi vào đó,
đúng như yêu cầu nộp bài ("phần code cấu hình để huấn luyện được YOLO, không cần source code YOLO").

**Trước khi chạy: bật GPU** — Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ **T4 GPU**.
Nếu không bật, mỗi model sẽ mất hàng giờ thay vì vài phút.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chưa bật GPU: Runtime ▸ Change runtime type ▸ T4 GPU'

## 2. Cài Ultralytics và lấy code cấu hình

In [ ]:
!pip -q install ultralytics
!git clone -q https://github.com/vo-hoang-kh4ng/Leaf-Segmentation.git repo
%cd repo
!git log --oneline -1

## 2b. Ghim đường dẫn của Ultralytics

Ultralytics lưu `datasets_dir` và `runs_dir` trong một file cấu hình **toàn cục**, không theo
project. Nếu không ghim, kết quả có thể rơi ra ngoài thư mục repo và ô nén ở cuối sẽ ra file rỗng.

In [ ]:
from ultralytics import settings
settings.update({'datasets_dir': '/content/repo/datasets', 'runs_dir': '/content/repo/runs'})
print(settings['datasets_dir']); print(settings['runs_dir'])

## 3. Xem giao thức huấn luyện trước khi chạy

Ba cỡ model phải dùng **cùng một giao thức** thì bảng so sánh mới có nghĩa; toàn bộ tham số nằm
trong `src/yolo/config.py`. Ô dưới chỉ in kế hoạch, không chạy gì.

In [ ]:
!cat src/yolo/config.py | head -60
!python -m src.yolo.train --task detect --device 0 --imgsz 640 --epochs 50 --batch 16 --dry-run

## 4. Huấn luyện Object Detection — yolo11n, yolo11s, yolo11m

Dataset **African Wildlife** (4 lớp: buffalo, elephant, rhino, zebra) tự tải về, khoảng 100 MB.

Trên T4, cả ba model ở 640px mất khoảng 25–35 phút. Kết quả cuối được chấm trên split **test**,
không phải val, vì Ultralytics đã dùng val để chọn `best.pt`.

In [ ]:
!python -m src.yolo.train --task detect --device 0 --imgsz 640 --epochs 50 --batch 16

## 5. Huấn luyện Object Segmentation

Cùng script, chỉ đổi `--task segment`; nó tự dùng checkpoint `-seg` và dataset segmentation
khai báo trong `src/yolo/config.py`.

In [ ]:
!python -m src.yolo.train --task segment --device 0 --imgsz 640 --epochs 50 --batch 16

## 6. Xem các hình minh chứng huấn luyện

Ultralytics tự sinh `results.png` (đường loss và mAP theo epoch), `confusion_matrix.png` và các
đường PR/F1 — đây chính là hình mà đề bài yêu cầu nộp.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
for task in ('detect', 'segment'):
    for run in sorted(Path('runs').glob(f'{task}/yolo11*')):
        results = run / 'results.png'
        if results.exists():
            print('=' * 70); print(run)
            display(Image(filename=str(results), width=950))

## 7. Bảng so sánh ba cỡ model

In [ ]:
import pandas as pd
for task in ('detect', 'segment'):
    path = Path(f'results/yolo_{task}_summary.csv')
    if path.exists():
        t = pd.read_csv(path)
        cols = [c for c in ('model','params_m','train_hours','mAP50(B)','mAP50-95(B)',
                            'mAP50(M)','mAP50-95(M)') if c in t.columns]
        print(f'--- {task} ---'); print(t[cols].to_string(index=False)); print()

## 8. Tải kết quả về máy

Nén `runs/` và `results/` rồi tải về: trong đó có `results.csv`, các hình minh chứng và trọng số
`best.pt` của từng model. Phần này dùng để viết báo cáo và nộp bài.

In [ ]:
!zip -qr yolo_ketqua.zip runs results -x "*.pt"
!ls -lh yolo_ketqua.zip
from google.colab import files
files.download('yolo_ketqua.zip')

---
**Lưu ý:** file zip trên đã loại trọng số `.pt` cho nhẹ. Nếu cần `best.pt` để chạy suy luận sau này,
bỏ `-x "*.pt"` — khi đó dung lượng tăng vài chục MB mỗi model.